In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold, train_test_split

train = pd.read_csv('../data/raw/train.csv')
print(f"Shape: {train.shape}")

In [ ]:
# Drop outliers identified in EDA
train = train[~((train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000))]

# Log-transform target
y = np.log1p(train['SalePrice'])
X = train.drop(['SalePrice', 'Id'], axis=1)

print(f"Shape after dropping outliers: {train.shape}")
print(f"X: {X.shape}, y: {y.shape}")

In [ ]:
# 1. Total square footage
X['TotalSF'] = X['1stFlrSF'] + X['2ndFlrSF'] + X['TotalBsmtSF']

# 2. Total bathrooms (with weights)
#X['TotalBath'] = X['FullBath'] + 0.5 * X['HalfBath'] + X['BsmtFullBath'] + 0.5 * X['BsmtHalfBath']

# 3. Years since remodel
#X['RemodAge'] = X['YrSold'] - X['YearRemodAdd']

# 4. Has pool?
#X['HasPool'] = (X['PoolArea'] > 0).astype(int)

# 5. Has garage?
#X['HasGarage'] = (X['GarageArea'] > 0).astype(int)

# 6. Has basement?
#X['HasBsmt'] = (X['TotalBsmtSF'] > 0).astype(int)

In [ ]:
print(f"X shape: {X.shape}")
print(f"X columns last 5: {X.columns[-5:].tolist()}")
print(f"TotalSF in X: {'TotalSF' in X.columns}")
print(f"RemodAge in X: {'RemodAge' in X.columns}")

In [ ]:
print(f"After feature engineering: {X.shape}")
print(f"New columns: {X.columns[-6:].tolist()}")

In [ ]:
# 1. Fill numeric NaN with median
numeric_cols = X.select_dtypes(include=[np.number]).columns
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())

# 2. Fill categorical NaN with 'Missing' (to remove NaN)
cat_cols = X.select_dtypes(include=['object']).columns
X[cat_cols] = X[cat_cols].fillna('Missing')

# 3. One-hot encode categoricals
X = pd.get_dummies(X, drop_first=True)

print(f"After preprocessing: {X.shape}")
print(f"NaN count: {X.isnull().sum().sum()}")

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Model 1: Decision Tree
dt = DecisionTreeRegressor(random_state=42)
dt_scores = -cross_val_score(dt, X, y, cv=kf, scoring='neg_root_mean_squared_error')
print(f"Decision Tree RMSE: {dt_scores.mean():.4f} ± {dt_scores.std():.4f}")

# Model 2: Random Forest
rf = RandomForestRegressor(random_state=42, n_jobs=-1)
rf_scores = -cross_val_score(rf, X, y, cv=kf, scoring='neg_root_mean_squared_error')
print(f"Random Forest RMSE: {rf_scores.mean():.4f} ± {rf_scores.std():.4f}")

# Model 3: Ridge (with scaling)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
ridge = Ridge(alpha=1.0)
ridge_scores = -cross_val_score(ridge, X_scaled, y, cv=kf, scoring='neg_root_mean_squared_error')
print(f"Ridge RMSE: {ridge_scores.mean():.4f} ± {ridge_scores.std():.4f}")

In [ ]:
rf.fit(X, y)
importances = pd.Series(rf.feature_importances_, index=X.columns)
top = importances.sort_values(ascending=False).head(20)
print(top)

In [ ]:
print(f"Total cols: {len(X.columns)}")
print(f"All engineered features in X:")
for col in ['TotalSF', 'TotalBath', 'RemodAge', 'HasPool', 'HasGarage', 'HasBsmt']:
    print(f"  {col}: {col in X.columns}")